# AI 폰트 파이프라인 — 다중생성 + OCR 자동선택

**Run all 한 번** → 글자당 8 샘플 생성 → 한글 OCR 이 목표 글자로 읽은 것만 채택
→ **coverage 숫자** (9 글자 중 몇 개가 정답 샘플 보유) + 선택 그리드.

지금까지 검증된 최선 설정 baked-in: chara_nums=11172, cont/sk=3.0, ddim50,
학습정합 참조(1_train.png).  소요 ~12 분 (T4).

핵심 질문: **다중생성으로 모든 글자에 도달 가능한가?**
- coverage 9/9 → usable 폰트 경로 존재
- 낮음 → 이 ckpt 는 개념 데모, 더 강한 모델 학습 필요


## 1. 환경 + repo

In [ ]:
!nvidia-smi | head -12
import os
if not os.path.exists('Korean-Diff-Font'):
    !git clone -q https://github.com/ORI-Muchim/Korean-Diff-Font.git
%cd Korean-Diff-Font

## 2. 의존성 + attrdict 패치 + easyocr

In [ ]:
!apt-get install -y libopenmpi-dev 2>&1 | tail -2
!pip install -q tqdm opencv-python scikit-learn pillow tensorboardX 'blobfile>=1.0.5' mpi4py attrdict pyyaml easyocr 2>&1 | tail -3

# attrdict Python 3.12 호환 (from collections import Mapping → collections.abc)
import glob
for f in glob.glob('/usr/local/lib/python3.*/dist-packages/attrdict/*.py'):
    s = open(f).read()
    s2 = (s.replace('from collections import Mapping', 'from collections.abc import Mapping')
            .replace('from collections import MutableMapping', 'from collections.abc import MutableMapping')
            .replace('from collections import Sequence', 'from collections.abc import Sequence'))
    if s2 != s:
        open(f, 'w').write(s2)
import sys
for m in [k for k in sys.modules if k == 'attrdict' or k.startswith('attrdict.')]:
    del sys.modules[m]
from attrdict import AttrDict
print('deps + attrdict patch OK:', AttrDict({'k': 'v'}).k)

## 3. 체크포인트 (498 MB)

In [ ]:
!mkdir -p trained_models
!wget -q -O trained_models/ema_0.9999_800000.pt \
    https://github.com/ORI-Muchim/Korean-Diff-Font/releases/download/1.0/ema_0.9999_800000.pt
!ls -lh trained_models/

## 4. 학습정합 참조 + 생성 글자

In [ ]:
import urllib.request
from PIL import Image, ImageDraw, ImageFont

TARGET_FONT_URL = 'https://github.com/google/fonts/raw/main/ofl/nanumgothic/NanumGothic-Regular.ttf'
REF_CHAR = '가'
GEN_CHARS = '가나다라마바사아자'
urllib.request.urlretrieve(TARGET_FONT_URL, 'target_font.ttf')

# 학습(font2img.py) 정합: 128 캔버스, 폰트크기 100, 고정 오프셋 14, 슈퍼샘플 없음
def render_train(font_path, ch, canvas=128, size=100, off=14):
    f = ImageFont.truetype(font_path, size)
    img = Image.new('RGB', (canvas, canvas), (255, 255, 255))
    ImageDraw.Draw(img).text((off, off), ch, (0, 0, 0), font=f)
    return img

# 디스플레이용 (목표 행 렌더): ink-bbox 중앙정렬
def render_display(font_path, ch, size=128):
    px = size * 4
    f = ImageFont.truetype(font_path, int(px * 0.7))
    img = Image.new('L', (px, px), 255)
    d = ImageDraw.Draw(img)
    b = d.textbbox((0, 0), ch, font=f)
    d.text(((px - (b[2] - b[0])) / 2 - b[0], (px - (b[3] - b[1])) / 2 - b[1]), ch, font=f, fill=0)
    return img.resize((size, size), Image.LANCZOS)

render_train('target_font.ttf', REF_CHAR).save('1_train.png')
print('참조 1_train.png + GEN_CHARS', GEN_CHARS, '준비')

## 5. cfg + 생성 대상 (글자당 8 샘플)

In [ ]:
N = 8
cfg = f'''model_path: ./trained_models/ema_0.9999_800000.pt
sty_img_path: ./1_train.png
total_txt_file: ./total_kor.txt
gen_txt_file: ./gen_char_kor.txt
stroke_path: ./korean_stroke.txt
img_save_path: ./result_pool
chara_nums: 11172
image_size: 128
num_channels: 128
num_res_blocks: 3
attention_resolutions: "40,20,10"
dropout: 0.1
diffusion_steps: 1000
noise_schedule: linear
use_ddim: true
timestep_respacing: "ddim50"
classifier_free: true
cont_scale: 3.0
sk_scale: 3.0
batch_size: 1
num_samples: {len(GEN_CHARS) * N}
'''
import os
os.makedirs('cfg', exist_ok=True)
open('cfg/pool.yaml', 'w').write(cfg)
open('gen_char_kor.txt', 'w', encoding='utf-8').write(''.join(c * N for c in GEN_CHARS))
print(f'cfg: {len(GEN_CHARS)} 글자 x {N} = {len(GEN_CHARS) * N} 샘플, cont/sk=3.0 ddim50')

## 6. 생성 (~7 분)

In [ ]:
import subprocess, shutil, os
if os.path.exists('result_pool'):
    shutil.rmtree('result_pool')
os.makedirs('result_pool')
subprocess.run(['python', 'sample.py', '--cfg_path', 'cfg/pool.yaml'], check=False)
pool_pngs = sorted(f for f in os.listdir('result_pool') if f.endswith('.png'))
print(len(pool_pngs), 'PNG 생성')

## 7. OCR 판독 + coverage

In [ ]:
import easyocr, numpy as np
from PIL import Image

reader = easyocr.Reader(['ko'], gpu=True)

def ocr(path, m=48):
    im = Image.open(path).convert('L')
    c = Image.new('L', (im.width + 2 * m, im.height + 2 * m), 255)
    c.paste(im, (m, m))
    return ''.join(reader.readtext(np.array(c), detail=0)).strip()

N = 8
pool = {}
for r, ch in enumerate(GEN_CHARS):
    es = []
    for s in range(N):
        i = r * N + s
        if i < len(pool_pngs):
            p = f'result_pool/{pool_pngs[i]}'
            t = ocr(p)
            es.append((p, t, t == ch))
    pool[ch] = es
    print(f'{ch}: {sum(m for _, _, m in es)}/{N} | {[t or "·" for _, t, _ in es]}')

cov = sum(1 for ch in GEN_CHARS if any(m for _, _, m in pool[ch]))
print(f'\n=== coverage: {cov}/{len(GEN_CHARS)} 글자가 ≥1 정답 샘플 보유 ===')

## 8. 선택 그리드 + 최종 폰트 후보

In [ ]:
from PIL import Image, ImageDraw

N = 8
CELL, PAD, LABELW, TEXTH = 96, 6, 80, 20
W = LABELW + (CELL + PAD) * (N + 1) + PAD
ROWH = CELL + TEXTH + PAD
out = Image.new('RGB', (W, ROWH * len(GEN_CHARS) + PAD), (255, 255, 255))
draw = ImageDraw.Draw(out)
for r, ch in enumerate(GEN_CHARS):
    y = PAD + r * ROWH
    draw.text((4, y + CELL // 2), f'{ch} 목표', fill=(0, 0, 0))
    out.paste(render_display('target_font.ttf', ch, CELL).convert('RGB'), (LABELW, y))
    for s, (p, txt, match) in enumerate(pool[ch]):
        x = LABELW + (s + 1) * (CELL + PAD)
        im = Image.open(p).convert('RGB').resize((CELL, CELL))
        out.paste(im, (x, y))
        col = (0, 170, 0) if match else (220, 60, 60)
        draw.rectangle([x, y, x + CELL, y + CELL], outline=col, width=3)
        draw.text((x + 2, y + CELL + 2), f'OCR:{txt or "·"}', fill=col)
out.save('ocr_select.png')
print('ocr_select.png — 녹색=OCR 목표 일치(선택)')
display(out)

# 최종: 글자당 선택 1 샘플 (다중생성+선택 폰트 후보)
sel = Image.new('RGB', (LABELW + (CELL + PAD) * len(GEN_CHARS) + PAD, CELL + 2 * PAD), (255, 255, 255))
d2 = ImageDraw.Draw(sel)
d2.text((4, CELL // 2), '선택 결과', fill=(0, 0, 0))
for c, ch in enumerate(GEN_CHARS):
    picks = [p for p, t, m in pool[ch] if m]
    x = LABELW + c * (CELL + PAD)
    if picks:
        im = Image.open(picks[0]).convert('RGB').resize((CELL, CELL)); box = (0, 170, 0)
    else:
        im = Image.new('RGB', (CELL, CELL), (245, 245, 245)); box = (220, 60, 60)
    sel.paste(im, (x, PAD))
    d2.rectangle([x, PAD, x + CELL, PAD + CELL], outline=box, width=3)
sel.save('selected_font.png')
print('selected_font.png — 글자당 선택 1 샘플 (실패=회색)')
display(sel)